In [1]:
# Author: Gergely Zahoranszky-Kohalmi, PhD
#
# Email: gergely.zahoranszky-kohalmi@nih.gov
#
# Organization: National Center for Advancing Translational Sciences
#


In [2]:
# environment: routesim

import pandas as pd

import os

import requests
import json

import rxnutils
from rxnutils import chem
from rxnutils.chem import utils
from rxnutils.chem.utils import remove_atom_mapping


In [3]:

FNAME_IN_EVIDENCE_ALL = []



DIR_IN_EVIDENCE = '../data/output/routes/rxns/'
DIR_OUT_EVIDENCE = '../data/output/routes/mapped_rxns/'

DO_SANITIZE = False
DO_CANONICALIZE = False

URL_RXNMAPPER = 'http://localhost:8002/syngps-app/api/v1/reaction_utils/atommap'



In [4]:
# Functions

def rxsmiles2mappedrxsmiles (rxsmiles):
    """
        Search depths: maximal number of reactions steps to explore.
    """
    
    # Base URL for your API

    rxsmiles = rxsmiles.split(' ')[0].strip()   # This returns the extension-free RXSMILES

    print (f'[*] Original RXSMILES: {rxsmiles} .')

    contituents = []
    contituents = rxsmiles.split('>')

    reactants = contituents[0].strip().split('.')
    reagents = contituents[1].strip().split('.')
    products = contituents[2].strip().split('.')





    
    no_map_reactants = []
    no_map_reagents = []
    no_map_products = []
    
    for smi in reactants:
        nomap_smi = remove_atom_mapping(smi, is_smarts = False, sanitize = DO_SANITIZE, canonical = DO_CANONICALIZE)
        no_map_reactants.append(nomap_smi)
    
    if len (reagents) > 0:
        for smi in reagents:
            nomap_smi = remove_atom_mapping(smi, is_smarts = False, sanitize = DO_SANITIZE, canonical = DO_CANONICALIZE)
            no_map_reagents.append(nomap_smi)

    for smi in products:
        nomap_smi = remove_atom_mapping(smi, is_smarts = False, sanitize = DO_SANITIZE, canonical = DO_CANONICALIZE)
        no_map_products.append(nomap_smi)
    
    reactants = ".".join(no_map_reactants)
    reagents = ".".join(no_map_reagents)
    products = ".".join(no_map_products)

    nomap_rxsmiles = ">".join([reactants, reagents, products])

    print (f'[*] Atom-mapping removed RXSMILES: {nomap_rxsmiles}')
    
    try:
        # Step 1: Make POST request to /api/test
        print("Atommapping reaction ...")
        
        # Optional: Include data in the first request if needed




        rxnmapper_payload = {
            "smiles": nomap_rxsmiles
        }



        # synth_route_search_payload = {
        #     "target_molecule_inchikey": target_molecule_inchikey,
        #     "reaction_steps": search_depth,
        #     "query_type": search_type,
        #     "leaves_as_sm": leaves_as_sm,
        #     "include_availability_info": False,
        #     "annotate_reactions": False,
        #     "graph_backend": "memgraph",
        #     "top_n_routes": top_n,
        #     "include_route_candidates": False,
        #     "include_combination_graphs": False
        # }
        
        response1 = requests.post(
            url = URL_RXNMAPPER,
            json = rxnmapper_payload,
            headers={'Content-Type': 'application/json',
                     'accept': 'application/json'})
        
        # Check if the request was successful
        #response1.raise_for_status()
        
        # Get JSON data from response
        received_data = response1.json()
        #print(f"Received data: {json.dumps(received_data, indent=2)}")


        return (received_data['mapped_rxsmiles'])
    
    except requests.exceptions.RequestException as e:
    
        print(f"Error occurred: {e}")

        return (None)


def annotate_routes_by_atommapping (fname_in, fname_out):
    df = pd.read_csv (fname_in, sep = '\t')

    print (df)

    df['real_mapped_rxsmiles'] = df.apply(lambda x: rxsmiles2mappedrxsmiles (x['rxsmiles']), axis = 1)

    df = df[['tm_inchikey', 'rxsmiles', 'route_index', 'real_mapped_rxsmiles']].copy()

    df = df.rename (columns = {
        'real_mapped_rxsmiles': 'mapped_reaction_smiles'
    })

    df.to_csv(fname_out, sep = '\t', index = False)






In [5]:
# Ensure DIR_OUT_EVIDENCE exists
if not os.path.exists(DIR_OUT_EVIDENCE):
    os.makedirs(DIR_OUT_EVIDENCE)

search_obj = os.scandir(DIR_IN_EVIDENCE)

for dir_item in search_obj:
    if dir_item.is_file():
        
        FNAME_IN_EVIDENCE_ALL.append(DIR_IN_EVIDENCE + dir_item.name)

print(FNAME_IN_EVIDENCE_ALL)

idx = 1

for fname in FNAME_IN_EVIDENCE_ALL:
    fname_out = fname.split('/')[-1]                    \
                                    .strip()            \
                                    .split('.')[0]      \
                                    .strip()          \
                                    + '_evidence_mapped_rxn.tsv'
    
    fname_out = DIR_OUT_EVIDENCE + fname_out
    
    print (f'[*] Processing route nr. {idx} of {len(FNAME_IN_EVIDENCE_ALL)} ..')
    
    idx += 1

    try:
        annotate_routes_by_atommapping (fname, fname_out)
    
    except:
        print (f'[W] Some of the reactions could not be atommapped in input file: {fname} .')

    print (f'[*] .. done.')


['../data/output/routes/rxns/OTNJGCAOEGZZMM-UHFFFAOYSA-N_evidence_based_route_rxn.tsv', '../data/output/routes/rxns/IHXNMNXXLBANRT-UHFFFAOYSA-N_evidence_based_route_rxn.tsv', '../data/output/routes/rxns/HLXQFVUPAUJHJI-VIFPVBQESA-N_evidence_based_route_rxn.tsv', '../data/output/routes/rxns/LAPOAOVLRHXODL-ZWKOTPCHSA-N_evidence_based_route_rxn.tsv', '../data/output/routes/rxns/YMGSLNVZHUIQNC-UHFFFAOYSA-N_evidence_based_route_rxn.tsv', '../data/output/routes/rxns/INSNUPHZRXHOTA-UHFFFAOYSA-N_evidence_based_route_rxn.tsv', '../data/output/routes/rxns/MWRHOSKPSCHFLP-UHFFFAOYSA-N_evidence_based_route_rxn.tsv', '../data/output/routes/rxns/VFBILHPIHUPBPZ-UHFFFAOYSA-N_evidence_based_route_rxn.tsv', '../data/output/routes/rxns/GZYNUOMXGGSXMI-UHFFFAOYSA-N_evidence_based_route_rxn.tsv', '../data/output/routes/rxns/AISQMYCMHKTHHN-SSEXGKCCSA-N_evidence_based_route_rxn.tsv', '../data/output/routes/rxns/QIXXLCJOJGHZJB-UHFFFAOYSA-N_evidence_based_route_rxn.tsv', '../data/output/routes/rxns/OJUKGYKYSNNNSA

In [6]:
print ('[Done.]')

[Done.]


In [7]:
# References:

# Ref: https://github.com/rxn4chemistry/rxnmapper
#

